[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/17_multi_armed_bandits/notebooks/01_exploracion_basica.ipynb)

# Notebook 1: El Problema del Bandido Multibrazo

**Módulo 17 — Multi-Armed Bandits · Notebook 01**

---

## ¿De qué trata este notebook?

En este notebook construimos desde cero el entorno clásico del **bandido multibrazo**
(multi-armed bandit) y exploramos las tensiones fundamentales entre **exploración** y
**explotación**.

**Lo que vas a construir y explorar:**

1. La clase `BanditEnvironment` que simula bandidos con recompensas Bernoulli o Gaussianas.
2. Dos problemas canónicos: uno Bernoulli y uno Gaussiano.
3. Exploración manual: tirar de cada brazo y estimar medias muestrales.
4. Convergencia de la media muestral conforme aumenta el número de tiradas.
5. El concepto de **regret acumulado** y por qué importa.
6. Comparación de estrategias puras: explotación ciega, exploración uniforme y oráculo.
7. Un primer intento de estrategia "inteligente": explorar-luego-explotar.

**Relación con las notas de clase:** sección 01 (Formulación del problema y estrategias puras).

**Prerequisitos:** Python básico (clases), numpy, matplotlib, probabilidad básica.

---

In [ ]:
# Solo en Colab — en entorno local estas librerías ya deben estar instaladas
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "red": "#E94F37", "orange": "#F28C28", "blue": "#2E86AB",
    "green": "#27AE60", "purple": "#8E44AD", "gray": "#7F8C8D",
}
ARM_COLORS = [COLORS["red"], COLORS["orange"], COLORS["blue"]]
ARM_NAMES = ["A", "B", "C"]

np.random.seed(42)

---

## 1. La clase `BanditEnvironment`

Un **bandido multibrazo** (multi-armed bandit) es un modelo de decisión secuencial
donde un agente debe elegir entre $K$ acciones ("brazos") en cada ronda. Al tirar
del brazo $k$, recibe una recompensa aleatoria cuya distribución depende del brazo
pero es **desconocida** para el agente.

Formalmente:
- Hay $K$ brazos con medias $\mu_1, \mu_2, \dots, \mu_K$.
- El brazo óptimo es $k^{∗} = \arg\max_k \mu_k$ con recompensa esperada $\mu^{∗}$.
- El **regret instantáneo** de tirar del brazo $k$ es $\Delta_k = \mu^{∗} - \mu_k$.
- El **regret acumulado** tras $T$ rondas es $R_T = \sum_{t=1}^{T} \Delta_{A_t}$.

Nuestra clase soporta dos tipos de recompensa:
- **Bernoulli**: $r \sim \text{Bernoulli}(\mu_k)$, donde $\mu_k \in [0,1]$.
- **Gaussiana**: $r \sim \mathcal{N}(\mu_k, \sigma^2)$.

In [ ]:
class BanditEnvironment:
    """Multi-armed bandit environment with Bernoulli or Gaussian rewards."""

    def __init__(self, mus, reward_type="bernoulli", sigma=1.5):
        """
        Parameters
        ----------
        mus : list[float]
            True mean reward for each arm.
        reward_type : str
            "bernoulli" or "gaussian".
        sigma : float
            Standard deviation (only used for Gaussian rewards).
        """
        self.mus = np.array(mus, dtype=float)
        self.reward_type = reward_type.lower()
        self.sigma = sigma
        self._best = np.argmax(self.mus)

    @property
    def K(self):
        """Number of arms."""
        return len(self.mus)

    def pull(self, arm):
        """Pull an arm and return a stochastic reward."""
        if self.reward_type == "bernoulli":
            return float(np.random.rand() < self.mus[arm])
        else:  # gaussian
            return np.random.normal(self.mus[arm], self.sigma)

    def optimal_reward(self):
        """Expected reward of the best arm."""
        return self.mus[self._best]

    def regret(self, arm):
        """Instantaneous regret of pulling the given arm."""
        return self.optimal_reward() - self.mus[arm]

    def __repr__(self):
        return (
            f"BanditEnvironment(K={self.K}, type={self.reward_type}, "
            f"mus={list(self.mus)}, best=arm {self._best})"
        )

---

## 2. Dos problemas canónicos

Trabajaremos con dos instancias a lo largo del notebook:

- **Problema A (Bernoulli):** Tres brazos con probabilidades de éxito $\mu = [0.3, 0.5, 0.7]$.
  El brazo C es el óptimo.
- **Problema B (Gaussiano):** Tres brazos con medias $\mu = [1.0, 2.0, 3.0]$ y $\sigma = 1.5$.
  El brazo C es el óptimo.

La diferencia entre brazos (el "gap" $\Delta_k$) determina qué tan difícil es
distinguir al mejor brazo.

In [ ]:
# Problema A: Bernoulli
bandit_A = BanditEnvironment(mus=[0.3, 0.5, 0.7], reward_type="bernoulli")

# Problema B: Gaussiano
bandit_B = BanditEnvironment(mus=[1.0, 2.0, 3.0], reward_type="gaussian", sigma=1.5)

print("Problema A:", bandit_A)
print(f"  Recompensa óptima: {bandit_A.optimal_reward()}")
for k in range(bandit_A.K):
    print(f"  Brazo {ARM_NAMES[k]}: mu={bandit_A.mus[k]:.1f}, gap={bandit_A.regret(k):.2f}")

print()
print("Problema B:", bandit_B)
print(f"  Recompensa óptima: {bandit_B.optimal_reward()}")
for k in range(bandit_B.K):
    print(f"  Brazo {ARM_NAMES[k]}: mu={bandit_B.mus[k]:.1f}, gap={bandit_B.regret(k):.2f}")

---

## 3. Exploración manual

Antes de diseñar algoritmos, experimentemos manualmente. Tiramos de cada brazo
$N$ veces y calculamos la **media muestral** $\hat{\mu}_k$.

Si $N$ es pequeño, las estimaciones serán ruidosas. ¿Cuántas muestras necesitamos
para distinguir al mejor brazo con confianza?

**Ejercicio:** Cambia el valor de `N` y observa cómo cambian las estimaciones.

In [ ]:
np.random.seed(42)

N = 30  # <-- CHANGE THIS: try 5, 30, 100, 500

env = bandit_A  # Usamos el problema Bernoulli

print(f"Tirando de cada brazo N={N} veces\n")
for k in range(env.K):
    rewards = [env.pull(k) for _ in range(N)]
    sample_mean = np.mean(rewards)
    print(
        f"Brazo {ARM_NAMES[k]}: "
        f"media muestral = {sample_mean:.3f}, "
        f"media real = {env.mus[k]:.3f}, "
        f"error = {abs(sample_mean - env.mus[k]):.3f}"
    )

---

## 4. Convergencia de la media muestral

Por la **Ley de los Grandes Números**, la media muestral $\hat{\mu}_k$ converge
a la media real $\mu_k$ conforme $N \to \infty$. Pero la velocidad de convergencia
depende de la varianza de las recompensas.

Veamos esto gráficamente: para cada brazo, tiramos 500 veces y graficamos la media
acumulada paso a paso.

In [ ]:
np.random.seed(42)

T = 500
env = bandit_A

fig, ax = plt.subplots(figsize=(10, 6))

for k in range(env.K):
    rewards = np.array([env.pull(k) for _ in range(T)])
    running_mean = np.cumsum(rewards) / np.arange(1, T + 1)
    ax.plot(
        running_mean,
        color=ARM_COLORS[k],
        label=f"Brazo {ARM_NAMES[k]} (est.)",
        alpha=0.8,
    )
    ax.axhline(
        env.mus[k],
        color=ARM_COLORS[k],
        linestyle="--",
        alpha=0.5,
        label=f"Brazo {ARM_NAMES[k]} (real = {env.mus[k]:.1f})",
    )

ax.set_xlabel("Número de tiradas")
ax.set_ylabel("Media muestral acumulada")
ax.set_title("Convergencia de la media muestral — Problema A (Bernoulli)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### Reflexión

Observa la gráfica anterior y piensa:

1. **¿Cuántas tiradas necesitas** para que la media muestral de cada brazo esté
   consistentemente cerca de la media real?
2. **¿El brazo con mayor varianza** (en este caso Bernoulli con $\mu=0.5$, que tiene
   varianza $0.25$) converge más lento o más rápido que los demás?
3. Si los gaps $\Delta_k$ fueran más pequeños (e.g., $\mu = [0.48, 0.50, 0.52]$),
   ¿sería más fácil o más difícil distinguir al mejor brazo?

Esta tensión entre **precisión de las estimaciones** y **costo de explorar** es
exactamente el dilema exploración-explotación.

---

## 5. Regret acumulado: midiendo el costo de aprender

El **regret acumulado** mide cuánta recompensa hemos perdido por no haber
seleccionado siempre al brazo óptimo:

$$R_T = \sum_{t=1}^{T} \left( \mu^{∗} - \mu_{A_t} \right)$$

donde $A_t$ es el brazo elegido en la ronda $t$.

Comparemos tres estrategias puras:

| Estrategia | Descripción | Regret esperado |
|---|---|---|
| **Explotación ciega** | Siempre elige brazo 0 (sin explorar) | $T \cdot \Delta_0$ (lineal) |
| **Exploración uniforme** | Round-robin entre los $K$ brazos | $T \cdot \bar{\Delta}$ (lineal) |
| **Oráculo** | Siempre elige el brazo óptimo $k^{∗}$ | $0$ |

Un buen algoritmo debería tener regret **sublineal** en $T$ — crece, pero cada vez
más lento.

In [ ]:
def compute_cumulative_regret(env, arm_sequence):
    """Compute cumulative regret for a given sequence of arm choices."""
    instantaneous = np.array([env.regret(a) for a in arm_sequence])
    return np.cumsum(instantaneous)


np.random.seed(42)

T = 1000
env = bandit_A

# Strategy 1: Pure exploitation — always pull arm 0
exploit_arms = np.zeros(T, dtype=int)

# Strategy 2: Pure exploration — round-robin
explore_arms = np.array([t % env.K for t in range(T)])

# Strategy 3: Oracle — always pull the best arm
oracle_arms = np.full(T, env._best, dtype=int)

regret_exploit = compute_cumulative_regret(env, exploit_arms)
regret_explore = compute_cumulative_regret(env, explore_arms)
regret_oracle = compute_cumulative_regret(env, oracle_arms)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(regret_exploit, color=COLORS["red"], label="Explotación ciega (brazo 0)", linewidth=2)
ax.plot(regret_explore, color=COLORS["orange"], label="Exploración uniforme (round-robin)", linewidth=2)
ax.plot(regret_oracle, color=COLORS["green"], label="Oráculo (siempre brazo óptimo)", linewidth=2)

ax.set_xlabel("Ronda $t$")
ax.set_ylabel("Regret acumulado $R_t$")
ax.set_title("Regret acumulado — Estrategias puras (Problema A)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Regret final — Explotación ciega: {regret_exploit[-1]:.1f}")
print(f"Regret final — Exploración uniforme: {regret_explore[-1]:.1f}")
print(f"Regret final — Oráculo: {regret_oracle[-1]:.1f}")

---

## 6. Ejercicio: Explorar-luego-explotar

Una idea natural es hacer **dos fases**:
1. **Exploración**: tirar de cada brazo $M$ veces para estimar las medias.
2. **Explotación**: usar el brazo con mayor media muestral durante el resto.

**Tu tarea:** completa la función `explore_then_exploit` y experimenta con
distintos valores de `M`.

**Preguntas:**
- ¿Qué pasa si $M$ es muy pequeño? ¿Y si es muy grande?
- ¿Existe un $M$ óptimo? ¿Cómo depende de $T$?

In [ ]:
def explore_then_exploit(env, T, M):
    """Explore-then-exploit strategy.

    Phase 1: pull each arm M times (total K*M pulls).
    Phase 2: pull the arm with highest sample mean for the remaining pulls.

    Returns the sequence of arm choices (length T).
    """
    arm_sequence = []
    counts = np.zeros(env.K)
    totals = np.zeros(env.K)

    # Phase 1: exploration
    for k in range(env.K):
        for _ in range(M):
            reward = env.pull(k)
            counts[k] += 1
            totals[k] += reward
            arm_sequence.append(k)

    # Phase 2: exploitation — pick the empirical best
    sample_means = totals / counts
    best_hat = np.argmax(sample_means)
    remaining = T - len(arm_sequence)
    arm_sequence.extend([best_hat] * remaining)

    return np.array(arm_sequence[:T])


np.random.seed(42)

T = 1000
env = bandit_A

M = 10  # <-- CHANGE THIS: try 5, 10, 30, 100, 200

ete_arms = explore_then_exploit(env, T, M)
regret_ete = compute_cumulative_regret(env, ete_arms)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(regret_exploit, color=COLORS["red"], label="Explotación ciega", alpha=0.5, linewidth=1.5)
ax.plot(regret_explore, color=COLORS["orange"], label="Exploración uniforme", alpha=0.5, linewidth=1.5)
ax.plot(regret_oracle, color=COLORS["green"], label="Oráculo", alpha=0.5, linewidth=1.5)
ax.plot(regret_ete, color=COLORS["purple"], label=f"Explorar-luego-explotar (M={M})", linewidth=2.5)

# Mark the transition point
exploration_end = env.K * M
if exploration_end < T:
    ax.axvline(exploration_end, color=COLORS["gray"], linestyle=":", alpha=0.7,
               label=f"Fin exploración (t={exploration_end})")

ax.set_xlabel("Ronda $t$")
ax.set_ylabel("Regret acumulado $R_t$")
ax.set_title(f"Explorar-luego-explotar con M={M} (Problema A)")
ax.legend()
plt.tight_layout()
plt.show()

chosen_best = ete_arms[exploration_end] if exploration_end < T else -1
print(f"Brazo elegido tras exploración: {ARM_NAMES[chosen_best]}")
print(f"Regret final: {regret_ete[-1]:.1f}")

### Reflexión final

Con la estrategia explorar-luego-explotar observamos que:

1. **Si $M$ es muy pequeño**, las estimaciones son malas y podemos elegir el brazo
   equivocado para la fase de explotación — el regret crece linealmente con
   pendiente $\Delta_{\hat{k}}$.
2. **Si $M$ es muy grande**, identificamos bien al mejor brazo, pero desperdiciamos
   muchas rondas explorando brazos subóptimos.
3. **El regret siempre es lineal** con esta estrategia: o lineal por mala estimación,
   o lineal por exceso de exploración.

Esto motiva la pregunta central del módulo:

> ¿Es posible lograr regret **sublineal** — es decir, que crezca más lento que $T$?

La respuesta es **sí**, y los algoritmos que lo logran (UCB, Thompson Sampling)
alcanzan regret $O(\log T)$. En el siguiente notebook construiremos estos algoritmos.

---